In [1]:
import pandas as pd
import numpy as np
import optuna
import time
import warnings
from pathlib import Path
from numba.cpython.numbers import NAN

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)
from lightgbm import LGBMRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error, mean_absolute_percentage_error
NOTEBOOK_DIR = Path().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent
df = pd.read_csv(PROJECT_ROOT / "data" / "processed" / "dataset_full.csv")
print(f"Размер датасета: {df.shape}")

Размер датасета: (5930, 27)


In [2]:
print(df.columns)
mask = df['price'] <= df['price'].quantile(0.99)
clean_df = df[mask]
clean_df['house_segment'] = pd.cut(clean_df['max_floor'],
                                   bins=[0, 5, 10, 19, 100],
                                   labels=['low', 'standard', 'modern', 'high'])
print(clean_df['house_segment'])

Index(['rooms', 'district', 'year', 'material', 'price', 'total_area',
       'living_area', 'kitchen_area', 'current_floor', 'max_floor', 'is_ready',
       'metro', 'mini_disctrict', 'price_mln', 'price_log', 'is_last_floor',
       'is_first_floor', 'living_ratio', 'kitchen_ratio', 'floor_ratio',
       'area_floor_interaction', 'district_ready', 'material_age',
       'area_ratio_to_district', 'floor_category', 'distance_to_center',
       'distance_to_metro'],
      dtype='str')
0       standard
1            low
2            low
3         modern
4            low
          ...   
5925         low
5926         low
5927         low
5928        high
5929         low
Name: house_segment, Length: 5870, dtype: category
Categories (4, str): ['low' < 'standard' < 'modern' < 'high']


In [3]:
cat_features = ['district', 'material', 'mini_disctrict', 'district_ready', 'material_age', 'floor_category', 'house_segment']
for col in cat_features:
    clean_df[col] = clean_df[col].fillna('unknown')
    clean_df[col] = clean_df[col].astype('category')

In [4]:
features = ['rooms', 'district', 'year', 'material', 'total_area',
       'living_area', 'kitchen_area', 'current_floor', 'max_floor', 'is_ready',
       'metro', 'mini_disctrict','is_last_floor',
       'is_first_floor', 'living_ratio', 'kitchen_ratio', 'floor_ratio',
       'area_floor_interaction', 'district_ready', 'material_age',
       'area_ratio_to_district', 'floor_category', 'distance_to_center',
       'distance_to_metro', 'house_segment']

In [5]:
X = clean_df[features]
y = clean_df['price_log']

In [6]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=True, random_state=42)
# Создание признака avg_price_m2
df_helping = X_train.copy()
y_train_exp = np.exp(y_train)
df_helping['mean_sq'] = y_train_exp / df_helping['total_area']
df_helping['house_segment'] = pd.cut(df_helping['max_floor'],
                                   bins=[0, 5, 10, 19, 100],
                                   labels=['low', 'standard', 'modern', 'high'])
mean_sq_dict = df_helping.groupby('district')['mean_sq'].mean().to_dict()
segment_map = (
    df_helping.groupby(['district', 'house_segment'], observed=True)['mean_sq'].mean().to_dict()
)
X_train['mean_sq2_segmented'] = X_train.set_index(['district', 'house_segment']).index.map(segment_map)
print(segment_map)
X_test['mean_sq2_segmented'] = (
    X_test.set_index(['district', 'house_segment'])
    .index.map(segment_map)
)

{('unknown', 'low'): 143894.61738646467, ('unknown', 'standard'): 172899.05915831964, ('unknown', 'modern'): 208309.85481143236, ('unknown', 'high'): 213562.5627989124, ('Автозаводский район', 'low'): 133447.39051097812, ('Автозаводский район', 'standard'): 156513.11258866548, ('Автозаводский район', 'modern'): 195100.05241776106, ('Автозаводский район', 'high'): 187325.5679990964, ('Канавинский район', 'low'): 116776.22615932101, ('Канавинский район', 'standard'): 159176.87313441234, ('Канавинский район', 'modern'): 183657.6710015635, ('Канавинский район', 'high'): 240637.04214531227, ('Ленинский район', 'low'): 127932.67993013904, ('Ленинский район', 'standard'): 160942.54051899342, ('Ленинский район', 'modern'): 200235.55312774226, ('Ленинский район', 'high'): 202808.68343270323, ('Московский район', 'low'): 118993.05216578799, ('Московский район', 'standard'): 160792.75936257304, ('Московский район', 'modern'): 213022.84871167925, ('Московский район', 'high'): 194598.54113225872, (

In [7]:
print(X_train['floor_category'].value_counts())

floor_category
high    2021
mid     1513
low     1162
Name: count, dtype: int64


In [8]:
import lightgbm as lgb
train_data = lgb.Dataset(
    X_train,
    label = y_train,
    categorical_feature = cat_features
)
test_data = lgb.Dataset(
    X_test,
    label = y_test,
    categorical_feature=cat_features,
    reference=train_data
)
params = {
    'objective': 'regression',
    'metric': 'mape',
    'n_estimators': 1439,
    'max_depth': 7,
    'learning_rate': 0.09117761849001915,
    'reg_lambda': 32,
    'verbose': -1
}
model = lgb.train(
    params,
    train_data,
    valid_sets=[train_data, test_data],
    valid_names=['train', 'valid']
)

In [9]:

preds_log = model.predict(X_test)
true_pred = np.exp(preds_log)
true_price_test = np.exp(y_test)

# Исправленный порядок: сначала истинные значения, затем предсказания
r2 = r2_score(true_price_test, true_pred)
print(f"R2: {r2:.4f}")

mape = mean_absolute_percentage_error(true_price_test, true_pred)
mae = mean_absolute_error(true_price_test, true_pred)

print(f"MAPE: {mape * 100:.2f}%")
print(f"MAE:  {mae:,.2f} ₽")

R2: 0.8879
MAPE: 12.21%
MAE:  1,254,402.94 ₽


In [10]:
from sklearn.model_selection import StratifiedKFold
y_binned = pd.qcut(y, q=10, labels=False, duplicates='drop')
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fold_mape_scores = []
fold_r2_scores = []
fold_mae_scores = []
models = []
params = {
    'objective': 'regression',
    'metric': 'mape',
    'n_estimators': 1439,
    'max_depth': 7,
    'learning_rate': 0.0912,
    'reg_lambda': 32,
    'verbose': -1
}
print("Модель LGMRegressor")
# 3. Основной цикл кросс-валидации
for fold, (train_index, test_index) in enumerate(skf.split(X, y_binned)):
    print(f"\n--- Обучение фолда {fold + 1} ---")

    # Создаем выборки
    X_train_fold, X_val_fold = X.iloc[train_index], X.iloc[test_index]
    y_train_fold, y_val_fold = y.iloc[train_index], y.iloc[test_index]

    # Инициализируем и обучаем модель
    model = lgb.LGBMRegressor(**params)
    model.fit(
        X_train_fold,
        y_train_fold
    )

    # Предсказания и обратное преобразование
    preds_log = model.predict(X_val_fold)
    true_pred = np.exp(preds_log)
    true_price_test = np.exp(y_val_fold)

    # Расчет метрик
    mape = mean_absolute_percentage_error(true_price_test, true_pred)
    r2 = r2_score(true_price_test, true_pred)
    mae =  mean_absolute_error(true_price_test, true_pred)
    fold_mape_scores.append(mape)
    fold_mae_scores.append(mae)
    fold_r2_scores.append(r2)
    models.append(model)

    print(f"Фолд {fold + 1} | MAE: {mae:.1f} | MAPE: {mape * 100:.2f}% | R2: {r2:.4f}")

# 4. Итоговая статистика
print("\n" + "="*40)
print(f"Средний MAPE по фолдам: {np.mean(fold_mape_scores) * 100:.2f}% (± {np.std(fold_mape_scores) * 100:.2f}%)")
print(f"Средний MAE по фолдам: {np.mean(fold_mae_scores):.1f} (± {np.std(fold_mae_scores):.1f})")
print(f"Средний R2   по фолдам: {np.mean(fold_r2_scores):.4f} (± {np.std(fold_r2_scores):.4f})")

Модель LGMRegressor

--- Обучение фолда 1 ---
Фолд 1 | MAE: 1257304.2 | MAPE: 11.12% | R2: 0.8923

--- Обучение фолда 2 ---
Фолд 2 | MAE: 1202303.6 | MAPE: 11.53% | R2: 0.8893

--- Обучение фолда 3 ---
Фолд 3 | MAE: 1257460.6 | MAPE: 11.75% | R2: 0.9019

--- Обучение фолда 4 ---
Фолд 4 | MAE: 1260036.1 | MAPE: 11.25% | R2: 0.8934

--- Обучение фолда 5 ---
Фолд 5 | MAE: 1263251.6 | MAPE: 12.78% | R2: 0.8839

Средний MAPE по фолдам: 11.69% (± 0.59%)
Средний MAE по фолдам: 1248071.2 (± 22985.6)
Средний R2   по фолдам: 0.8922 (± 0.0059)


In [11]:
from catboost import CatBoostRegressor

model_catboost = CatBoostRegressor(
    iterations=1135,
    depth=8,
    learning_rate=0.06,
    l2_leaf_reg=27,
    random_seed=36,
    cat_features=['district', 'material', 'mini_disctrict', 'district_ready', 'material_age', 'floor_category', 'house_segment'],
    verbose=100,
    loss_function='MAPE',
    eval_metric='MAPE'
)

# Обучение модели
model_catboost.fit(X_train, y_train)

# Исправление: вызываем метод predict у объекта CatBoost
# 1. Получаем предсказания нужной модели
preds_catboost_log = model_catboost.predict(X_test)

# 2. Переводим в исходный масштаб (рубли)
preds_catboost_exp = np.exp(preds_catboost_log)
true_price_test = np.exp(y_test)

# 3. Убедитесь, что true_price_test также в исходном масштабе (рублях)
# (Если y_test был в логарифмах, то true_price_test = np.exp(y_test))

# 4. Вычисляем метрику
from sklearn.metrics import r2_score
r2_catboost = r2_score(true_price_test, preds_catboost_exp)
print(f"R2: {r2:.4f}")
mape_catboost = mean_absolute_percentage_error(true_price_test, preds_catboost_exp)
mae_catboost = mean_absolute_error(true_price_test, preds_catboost_exp)
print(f"MAPE: {mape_catboost * 100:.2f}%")
print(f"MAPE: {mae_catboost:.0f}рублей")

0:	learn: 0.0253798	total: 61.9ms	remaining: 1m 10s
100:	learn: 0.0078717	total: 606ms	remaining: 6.21s
200:	learn: 0.0063414	total: 1.16s	remaining: 5.39s
300:	learn: 0.0055129	total: 1.73s	remaining: 4.79s
400:	learn: 0.0050072	total: 2.31s	remaining: 4.24s
500:	learn: 0.0046115	total: 2.9s	remaining: 3.67s
600:	learn: 0.0043276	total: 3.51s	remaining: 3.12s
700:	learn: 0.0040928	total: 4.13s	remaining: 2.56s
800:	learn: 0.0039061	total: 4.72s	remaining: 1.97s
900:	learn: 0.0037515	total: 5.3s	remaining: 1.38s
1000:	learn: 0.0036100	total: 5.86s	remaining: 785ms
1100:	learn: 0.0034757	total: 6.42s	remaining: 198ms
1134:	learn: 0.0034298	total: 6.6s	remaining: 0us
R2: 0.8839
MAPE: 12.45%
MAPE: 1314315рублей


In [12]:
train_data = lgb.Dataset(
    X_train,
    label = y_train,
    categorical_feature = ['district', 'material', 'mini_disctrict', 'district_ready', 'material_age', 'floor_category', 'house_segment']
)
test_data = lgb.Dataset(
    X_test,
    label = y_test,
    categorical_feature=cat_features,
    reference=train_data
)
params = {
    'objective': 'regression',
    'metric': 'mape',
    'n_estimators': 1439,
    'max_depth': 7,
    'learning_rate': 0.09117761849001915,
    'reg_lambda': 32,
    'verbose': -1
}
model_lgb = lgb.train(
    params,
    train_data,
    valid_sets=[train_data, test_data],
    valid_names=['train', 'valid']
)

In [13]:
preds_lgb_log = model_lgb.predict(X_test)
preds_cat_log = model_catboost.predict(X_test)
final_preds_log = 0.5 * preds_lgb_log + 0.5 * preds_cat_log
true_pred = np.exp(final_preds_log)
true_price_test = np.exp(y_test)
r2 = r2_score(true_price_test, true_pred)
mape = mean_absolute_percentage_error(true_price_test, true_pred)
mae = mean_absolute_error(true_price_test, true_pred)

# 5. Выводим результаты
print(f"R2:   {r2:.4f}")
print(f"MAPE: {mape * 100:.2f}%")
print(f"MAE:  {mae:,.2f} ₽")

R2:   0.8917
MAPE: 11.89%
MAE:  1,232,047.57 ₽


In [14]:
# Инициализируем StratifiedKFold
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Списки для сохранения результатов каждого фолда
fold_mape_scores = []
fold_mae_scores = []
fold_r2_scores = []

# Параметры моделей
params_lgb = {
    'objective': 'regression',
    'metric': 'mape',
    'n_estimators': 1439,
    'max_depth': 7,
    'learning_rate': 0.09117761849001915,
    'reg_lambda': 32,
    'verbose': -1
}

params_cat = {
    'iterations': 1135,
    'depth': 8,
    'learning_rate': 0.06,
    'l2_leaf_reg': 27,
    'random_seed': 36,
    'cat_features': ['district', 'material', 'mini_disctrict', 'district_ready', 'material_age', 'floor_category', 'house_segment'],
    'verbose': False, # Можно поставить 100, чтобы видеть прогресс CatBoost
    'loss_function': 'MAPE',
    'eval_metric': 'MAPE'
}
print("Две взвешенные модели: LightGBM(с большим весом, 0,6) и CatBoost(0.4 вес)")
# Основной цикл
for fold, (train_index, test_index) in enumerate(skf.split(X, y_binned)):
    print(f"\n--- Обучение фолда {fold + 1} ---")

    # Разделение выборок
    X_train_fold, X_val_fold = X.iloc[train_index], X.iloc[test_index]
    y_train_fold, y_val_fold = y.iloc[train_index], y.iloc[test_index]

    # 1. Обучение LightGBM
    lgb_model = lgb.LGBMRegressor(**params_lgb)
    lgb_model.fit(X_train_fold, y_train_fold)

    # 2. Обучение CatBoost
    cat_model = CatBoostRegressor(**params_cat)
    cat_model.fit(X_train_fold, y_train_fold)

    # 3. Предсказания
    preds_lgb_log = lgb_model.predict(X_val_fold)
    preds_cat_log = cat_model.predict(X_val_fold)

    # 4. Усреднение предсказаний
    final_preds_log = 0.6 * preds_lgb_log + 0.4 * preds_cat_log

    # 5. Обратное преобразование
    true_pred = np.exp(final_preds_log)
    true_price_test = np.exp(y_val_fold)

    # 6. Расчет метрик
    r2 = r2_score(true_price_test, true_pred)
    mape = mean_absolute_percentage_error(true_price_test, true_pred)
    mae = mean_absolute_error(true_price_test, true_pred)

    fold_r2_scores.append(r2)
    fold_mape_scores.append(mape)
    fold_mae_scores.append(mae)

    print(f"Фолд {fold + 1} | MAE: {mae:,.2f} ₽ | MAPE: {mape * 100:.2f}% | R2: {r2:.4f}")

# Итоговая статистика
print("\n" + "="*40)
print(f"Средний MAPE: {np.mean(fold_mape_scores) * 100:.2f}% (± {np.std(fold_mape_scores) * 100:.2f}%)")
print(f"Средний MAE:  {np.mean(fold_mae_scores):,.2f} ₽ (± {np.std(fold_mae_scores):,.2f})")
print(f"Средний R2:   {np.mean(fold_r2_scores):.4f} (± {np.std(fold_r2_scores):.4f})")

Две взвешенные модели: LightGBM(с большим весом, 0,6) и CatBoost(0.4 вес)

--- Обучение фолда 1 ---
Фолд 1 | MAE: 1,236,950.09 ₽ | MAPE: 10.81% | R2: 0.8916

--- Обучение фолда 2 ---
Фолд 2 | MAE: 1,182,301.40 ₽ | MAPE: 11.34% | R2: 0.8918

--- Обучение фолда 3 ---
Фолд 3 | MAE: 1,197,297.59 ₽ | MAPE: 11.26% | R2: 0.9085

--- Обучение фолда 4 ---
Фолд 4 | MAE: 1,217,050.32 ₽ | MAPE: 10.76% | R2: 0.8963

--- Обучение фолда 5 ---
Фолд 5 | MAE: 1,231,229.87 ₽ | MAPE: 12.54% | R2: 0.8873

Средний MAPE: 11.34% (± 0.64%)
Средний MAE:  1,212,965.86 ₽ (± 20,548.98)
Средний R2:   0.8951 (± 0.0073)


In [15]:
import numpy as np
import pandas as pd
import lightgbm as lgb
from catboost import CatBoostRegressor

# 1. Параметры моделей
params_lgb = {
    'objective': 'regression',
    'metric': 'mape',
    'n_estimators': 1439,
    'max_depth': 7,
    'learning_rate': 0.09117761849001915,
    'reg_lambda': 32,
    'verbose': -1
}

params_cat = {
    'iterations': 1135,
    'depth': 8,
    'learning_rate': 0.06,
    'l2_leaf_reg': 27,
    'random_seed': 36,
    'cat_features': ['district', 'material', 'mini_disctrict', 'district_ready', 'material_age', 'floor_category', 'house_segment'],
    'verbose': 100,
    'loss_function': 'MAPE',
    'eval_metric': 'MAPE'
}
print("Финальный ансамбль двух моделе: LightLGM и CatBoost")
# 2. Инициализация и обучение моделей
model_lgb = lgb.LGBMRegressor(**params_lgb)
model_cat = CatBoostRegressor(**params_cat)

print("Обучение LightGBM...")
model_lgb.fit(X, y)

print("\nОбучение CatBoost...")
model_cat.fit(X, y)

print("\nМодели успешно обучены на всем датасете!")

# 3. Функция для предсказания цены одной квартиры
def predict_apartment_price(apartment_features, X_train_columns, X_train):
    """
    apartment_features: словарь с признаками квартиры
    X_train_columns: список колонок (features) в том порядке, в котором они были при обучении
    X_train: обучающий датасет (DataFrame) для синхронизации типов категорий
    """
    # Превращаем словарь в DataFrame (с одной строкой)
    df_input = pd.DataFrame([apartment_features])

    # Приводим к порядку колонок, на которых обучались
    df_input = df_input.reindex(columns=X_train_columns, fill_value=0)

    # --- Подготовка датасета для LightGBM ---
    df_lgb = df_input.copy()
    for col in params_cat['cat_features']:
        if col in df_lgb.columns:
            if pd.api.types.is_categorical_dtype(X_train[col]):
                val = df_lgb[col].iloc[0]
                if val in X_train[col].cat.categories:
                    df_lgb[col] = pd.Categorical([val], categories=X_train[col].cat.categories)
                else:
                    # Если категории нет в обучении, используем первую попавшуюся
                    df_lgb[col] = pd.Categorical([X_train[col].cat.categories[0]], categories=X_train[col].cat.categories)
            else:
                df_lgb[col] = df_lgb[col].astype(X_train[col].dtype)

    # --- Подготовка датасета для CatBoost ---
    df_cat = df_input.copy()
    for col in params_cat['cat_features']:
        if col in df_cat.columns:
            val = df_cat[col].iloc[0]
            # Защита от NaN или пустых значений
            if pd.isna(val) or val is None or val == 0:
                df_cat[col] = 'Unknown'
            else:
                df_cat[col] = str(val)

    # Получаем предсказания в логарифмическом масштабе
    pred_lgb_log = model_lgb.predict(df_lgb)
    pred_cat_log = model_cat.predict(df_cat)

    # Ансамбль: 60% LightGBM + 40% CatBoost
    final_pred_log = 0.6 * pred_lgb_log + 0.4 * pred_cat_log

    # Перевод из логарифма в реальную цену
    final_price = np.exp(final_pred_log[0])

    return final_price

# --- Пример использования ---
all_columns = X.columns.tolist()



Финальный ансамбль двух моделе: LightLGM и CatBoost
Обучение LightGBM...

Обучение CatBoost...
0:	learn: 0.0255200	total: 4.21ms	remaining: 4.77s
100:	learn: 0.0079769	total: 451ms	remaining: 4.62s
200:	learn: 0.0065215	total: 951ms	remaining: 4.42s
300:	learn: 0.0057136	total: 1.46s	remaining: 4.06s
400:	learn: 0.0051999	total: 1.98s	remaining: 3.63s
500:	learn: 0.0048367	total: 2.56s	remaining: 3.24s
600:	learn: 0.0045657	total: 3.17s	remaining: 2.81s
700:	learn: 0.0043415	total: 3.74s	remaining: 2.31s
800:	learn: 0.0041545	total: 4.27s	remaining: 1.78s
900:	learn: 0.0040169	total: 4.95s	remaining: 1.28s
1000:	learn: 0.0038901	total: 6.31s	remaining: 845ms
1100:	learn: 0.0037623	total: 7.01s	remaining: 216ms
1134:	learn: 0.0037219	total: 7.23s	remaining: 0us

Модели успешно обучены на всем датасете!


In [16]:
sample_apartment = {
    'rooms': 1,
    'district': 'Советский район',
    'year': 2,
    'total_area': 33.9,
    'living_area': 10.7,
    'kitchen_area': 15.1,
    'current_floor': 1,
    'max_floor': 15,
    'is_ready': 1,
    'metro': 12.5,
    'is_last_floor': 0,
    'is_first_floor': 1,
    'living_ratio': 0.305,
    'kitchen_ratio': 0.44,
    'floor_ratio': 0.0666,
    'area_floor_interaction': 33.9,
    'area_ratio_to_district': 2,
    'distance_to_center': 4.3,
    'distance_to_metro': 44,
    'material': 'монолитный железобетон',
    'mini_disctrict': 'unknown',
    'district_ready': 'Советский район_1',
    'material_age': 'монолитный железобетон_False',
    'floor_category': 'low',
    'house_segment': 'low',
    'mean_sq2_segmented': 151112,
}

predicted_price = predict_apartment_price(sample_apartment, all_columns, X)
print(f"\nПрогнозируемая стоимость квартиры: {predicted_price:,.2f} ₽")


Прогнозируемая стоимость квартиры: 7,431,179.15 ₽


In [17]:
for col in df.columns:
    print(df[col].value_counts())

rooms
2.0    2174
1.0    1793
3.0    1372
0.0     279
4.0     269
5.0      25
6.0       8
8.0       1
7.0       1
Name: count, dtype: int64
district
unknown                1631
Нижегородский район    1128
Советский район         806
Канавинский район       647
Автозаводский район     545
Сормовский район        335
Ленинский район         314
Приокский район         305
Московский район        219
Name: count, dtype: int64
year
0.0      1300
1.0        99
5.0        63
2.0        57
3.0        54
         ... 
102.0       1
90.0        1
112.0       1
127.0       1
124.0       1
Name: count, Length: 102, dtype: int64
material
unknown                                  2711
кирпич                                   1462
блок+утеплитель                           827
панель                                    598
монолитный железобетон                    268
шлакоблок                                  37
дерево                                     17
поризованный керамический блок              